# Perform a Layer by Layer forward pass on a model

This notebook is purely for debugging purposes. It is not meant to be run as a script. The goal is to perform a forward pass on a model layer by layer, and print the output of each layer, and compare it to the output of the RISC-V/C model.

TODO: Clean this up later, this is a mess right now.

In [15]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random
import csv

from sklearn.model_selection import train_test_split

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [16]:
# Load and preprocess MNIST dataset
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0  # Normalize the images to [0, 1]
x_train = np.expand_dims(x_train, -1)  # Add channel dimension
y_train = tf.keras.utils.to_categorical(y_train, 10)  # One-hot encode the labels

In [17]:
# Get a 1 random image
idx = random.randint(0, len(x_train) - 1)
image = x_train[idx].squeeze()  # (28, 28)
label = np.argmax(y_train[idx])  # Get label

print(label)

3


In [18]:
import numpy as np

def print_output_shape_and_values(x):
    print(f"Output shape: {x.shape}")
    
    # If it's a 4D tensor (e.g., batch of images), handle it
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                for j in range(width):
                    print(f"{x[0, i, j, c]:.3f}", end=" ")
                print()
            print()
        
    
    # If it's a 2D array (after flattening), handle it
    elif len(x.shape) == 2:
        # Extract height and width from flattened shape, assume one image
        rows, cols = x.shape
        for i in range(rows):
            for j in range(cols):
                print(f"{x[i, j]:.3f}", end=" ")
            print()
    else:
        print("Unsupported shape")


## Load the model from mnist_cnn_model.keras

In [19]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

In [20]:
x = tf.expand_dims(image, axis=0)  # Add batch dimension
x = tf.expand_dims(x, axis=-1)     # Add channel dimension

x = model.layers[0](x)  # First layer output

print_output_shape_and_values(x)

# Flatten the output and reshape it into 8 grids of 24x24
out = x.numpy().flatten()  # Flatten the output

# Reshape into 8 grids of 24x24
grids = out.reshape(8, 24, 24)

Output shape: (1, 24, 24, 8)
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.399 -0.241 -0.174 -0.135 -0.087 -0.225 -0.342 -0.486 -0.561 -0.514 -0.495 -0.464 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.366 -0.166 -0.065 -0.050 -0.041 0.055 0.187 0.274 0.330 0.242 -0.074 -0.356 -0.455 -0.509 -0.466 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.435 -0.265 -0.065 -0.026 -0.017 -0.012 0.065 0.150 0.190 0.252 0.387 0.484 0.409 0.127 -0.212 -0.403 -0.503 -0.466 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.330 -0.065 -0.034 -0.101 -0.151 -0.148 -0.010 -0.030 -0.152 -0.084 0.174 0.226 0.353 0.265 -0.002 -0.192 -0.402 -0.474 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.270 -0.244 -0.507 -0.481 -0.410 -0.264 -0.209 -0.302 -0.255 -0.369 -0.208 -0.075 0.002 0.074 -0.116 -0.112 -0.262 -0.414 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.502 -0.767 -0.836 -0.628 -0.357 -0.409 -0.498 -0.731 -0.674 -0.452 -0.397 -0.562 -0.455 -0.338 -0.167 -0.24

In [21]:
# Second Layer
x = model.layers[1](x)  # Second layer output
print_output_shape_and_values(x)

Output shape: (1, 24, 24, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.055 0.187 0.274 0.330 0.242 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.065 0.150 0.190 0.252 0.387 0.484 0.409 0.127 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.174 0.226 0.353 0.265 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.002 0.074 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000

In [22]:
# Third Layer
x = model.layers[2](x)  # Third layer output
print_output_shape_and_values(x)

Output shape: (1, 12, 12, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.187 0.330 0.242 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.065 0.190 0.387 0.484 0.265 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.002 0.074 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.141 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.016 0.000 0.120 0.000 0.000 
0.000 0.000 0.000 0.015 0.000 0.138 0.208 0.363 0.293 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.123 0.000 0.288 0.282 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.372 0.448 0.119 0.138 0.000 0.000 0.141 0.000 0.000 0.000 
0.000 0.000 0.461 0.473 0.189 0.091 0.000 0.000 0.130 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 

0.178 0.178 0.178 0.309 0.952 2.114 3.156 3.402 3.003 1.605 0.563 0.178 
0.178 0.178 0.375 1.3

In [23]:
# Fourth Layer
x = model.layers[3](x)  # Fourth layer output
print_output_shape_and_values(x)

Output shape: (1, 1152)
0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.309 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.952 0.000 0.000 0.113 0.177 0.363 0.000 0.000 2.114 0.000 0.000 0.229 0.152 0.610 0.000 0.187 3.156 0.000 0.000 0.349 0.000 0.770 0.000 0.330 3.402 0.000 0.000 0.114 0.000 0.864 0.000 0.242 3.003 0.126 0.000 0.000 0.000 0.497 0.000 0.000 1.605 0.092 0.278 0.000 0.000 0.000 0.000 0.000 0.563 0.000 0.196 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.375 0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.383 0.000 0.000 0.000 0.672 0.443 0.000 0.000 2.489 0.000 0.000 0.191 0.501 0.679 0.000 0.065 3.182 1.076 0.000 0.000 0.000 0.683 0.000 0.190 3.418 1.120 0.000 0.000 0.000 1.016 0.000 0.387 3.636 0.544 0.000 0.000 0.000 1.613 0.000 0.484 3.589 0.33

In [24]:
# Fifth Layer: Dense Layer
x = model.layers[4](x)  # Fifth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
-19.761 -9.140 -7.391 14.379 -29.236 -3.467 -25.492 -18.891 -5.843 -3.230 


In [25]:
# Sixth Layer: Softmax Layer
x = model.layers[5](x)  # Sixth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
0.000 0.000 0.000 1.000 0.000 0.000 0.000 0.000 0.000 0.000 


In [26]:
print(f"Predicted class: {np.argmax(x)}")
print(f"True class: {label}")

Predicted class: 3
True class: 3
